In [ ]:
import os

# **[FAST-AI]: Let’s Build the GPT Tokenizer: A Complete Guide to Tokenization in LLMs**
* **A text and code version of Karpathy’s famous tokenizer video.**

In [1]:
# import torch
text = 'This is some text dataset hello, and hi some words!'
# get the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

 !,Tadehilmnorstwx
18


In [2]:
# Step 1: Get the sample text from Nathan Reed's blog post
text = """Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to "support Unicode" in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don't blame programmers for still finding the whole thing mysterious, even 30 years after Unicode's inception."""

# print(f"Text: {text}")
print(f"Length in characters: {len(text)}")

Length in characters: 533


In [4]:
# Step 2: Encode the text to UTF-8 bytes and convert to list of integers
tokens = list(text.encode("utf-8"))
# print(f"UTF-8 encoded bytes: {tokens[:50]}...")  # Show first 50 bytes
print(f"Length in bytes: {len(tokens)}")

Length in bytes: 608


In [5]:
text.encode('utf-8')

b'\xef\xbc\xb5\xef\xbd\x8e\xef\xbd\x89\xef\xbd\x83\xef\xbd\x8f\xef\xbd\x84\xef\xbd\x85! \xf0\x9f\x85\xa4\xf0\x9f\x85\x9d\xf0\x9f\x85\x98\xf0\x9f\x85\x92\xf0\x9f\x85\x9e\xf0\x9f\x85\x93\xf0\x9f\x85\x94\xe2\x80\xbd \xf0\x9f\x87\xba\xe2\x80\x8c\xf0\x9f\x87\xb3\xe2\x80\x8c\xf0\x9f\x87\xae\xe2\x80\x8c\xf0\x9f\x87\xa8\xe2\x80\x8c\xf0\x9f\x87\xb4\xe2\x80\x8c\xf0\x9f\x87\xa9\xe2\x80\x8c\xf0\x9f\x87\xaa! \xf0\x9f\x98\x84 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to "support Unicode" in our software (whatever that means\xe2\x80\x94like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don\'t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode\'s inception.'

In [6]:
def get_stats(ids, counts=None):
    """
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [9]:
stats = get_stats(tokens)
# stats
len(stats)

267

In [12]:
# top_pair = stat
# max(stats, key = lambda p: stats.get(p, -1)) # (101, 32)
# stats[(101, 32)] # 20

20

In [14]:
top_pair = max(stats, key= stats.get)
print(top_pair)
print(stats[top_pair])

(101, 32)
20


In [19]:
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids) - 1:
        if (ids[i], ids[i+1]) == pair:
            newids += [idx]
            i += 2
        else:
            newids += [ids[i]]
            i += 1
    return newids

In [23]:
# "The BPE algorithm proceeds iteratively: find the most common pair, merge it, and repeat."
n_vocab = 276
n_merges = n_vocab - 256 # {2^8 = 256}

# BPE:
tokens_2 = list(tokens)
idx = 256
merges = {}
for i in range(n_merges):
    stats = get_stats(tokens_2)
    top_pair = max(stats, key=stats.get)
    if stats[top_pair] == 1:
        break
    merges[top_pair] = idx
    tokens_2 = merge(tokens_2, top_pair, idx)
    idx += 1

In [24]:
merges

{(101, 32): 256,
 (240, 159): 257,
 (105, 110): 258,
 (115, 32): 259,
 (97, 110): 260,
 (226, 128): 261,
 (116, 104): 262,
 (257, 133): 263,
 (257, 135): 264,
 (97, 114): 265,
 (239, 189): 266,
 (261, 140): 267,
 (267, 264): 268,
 (101, 114): 269,
 (111, 114): 270,
 (116, 32): 271,
 (258, 103): 272,
 (115, 116): 273,
 (260, 100): 274,
 (32, 262): 275}

In [51]:
# Decoding:
vocab = {i: bytes([i]) for i in range(255)}
for (p0, p1), id in merges.items():
    vocab[id] = vocab[p0] + vocab[p1]

def decode(ids: list) -> str:
    text = b"".join([vocab[id] for id in ids])
    text = text.decode('utf-8', errors='replace')
    return text

decode(tokens)

'Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺\u200c🇳\u200c🇮\u200c🇨\u200c🇴\u200c🇩\u200c🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to "support Unicode" in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don\'t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode\'s inception.'

In [58]:
merges

{(101, 32): 256,
 (240, 159): 257,
 (105, 110): 258,
 (115, 32): 259,
 (97, 110): 260,
 (226, 128): 261,
 (116, 104): 262,
 (257, 133): 263,
 (257, 135): 264,
 (97, 114): 265,
 (239, 189): 266,
 (261, 140): 267,
 (267, 264): 268,
 (101, 114): 269,
 (111, 114): 270,
 (116, 32): 271,
 (258, 103): 272,
 (115, 116): 273,
 (260, 100): 274,
 (32, 262): 275}

In [69]:
def encode(txt: str) -> list:
    tokens = list(txt.encode('utf-8'))
    # print(tokens)
    # print(f'Before : {len(tokens)}')
    while True:
        stats = get_stats(tokens)
        if len(tokens) < 2:
            break
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break
        tokens = merge(tokens, pair, merges[pair])
            
    # print(f'After : {len(tokens)}')
    return tokens

# encode('hello world how are you man this is way too good')
decode(encode('h'))

'h'

In [73]:
import regex as re
pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
text = "Hello world123 how are you?"
matches = pat.findall(text)
matches

['Hello', ' world', '123', ' how', ' are', ' you', '?']

In [74]:
pat.findall("     you")

['    ', ' you']

In [75]:
example = """
for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
"""

print(pat.findall(example))

['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', ':', '\n       ', ' print', '(', 'i', ')', '\n']
